# 16 — An Alignment Quantity for Whether a Transferred Ranking Survives

Notebook 15 showed that features whose association with the label reverses between corpora cause inversion, but that a count of such features does not predict which pairs invert. This notebook supplies the quantity that does, and tests both its exact form and the form an operator could actually compute.

For a linear scorer s(x) = sum_j w_j (x_j - mu_j) / sd_j standardised on the source, the covariance with the target label is sd_y times kappa = sum_j w_j (sd_target_j / sd_source_j) r_target_j, where r_target is the per-feature point-biserial correlation on the target. Since sd_y is positive, sign(kappa) is the sign of the correlation between transferred scores and target labels: negative kappa means an inverted ranking. Each feature enters weighted by the coefficient the source model places on it and by the ratio of spreads, which is exactly why a count of sign reversals fails as a predictor.

Two forms are evaluated. The oracle form estimates r_target and sd_target on the same evaluation data the score is computed on, which makes the relationship an identity and serves as a correctness check. The deployable form estimates them from the labelled buffer alone, at the budgets used throughout the paper, and asks whether a few hundred to a few thousand target labels are enough to predict the orientation of the deployed model before it is trusted.

Features with no variation in the source are dropped. They carry no source information, and after source standardisation their target values would be divided by an almost-zero spread and dominate the score for numerical reasons alone. The count is reported per corpus.

Results are written to fc_alignment.csv and fc_alignment_buffer.csv.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(corpora=['nf2018v2','nfunswv2','nftonv2','nfbotv2'], seed=42, test_size=0.30,
           train_cap=250_000, eval_cap=200_000, budgets=[0.0001, 0.001, 0.01],
           ece_bins=15, rf_estimators=300, mlp_hidden=(128,64), mlp_max_iter=100)
ALIGN_CSV = f'{RESULT}/fc_alignment.csv'
BUF_CSV   = f'{RESULT}/fc_alignment_buffer.csv'
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label','Attack')]
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np

def usable_features(X, rel=1e-9):
    """Mean, spread, and a mask of features the source model can legitimately use.

    A feature with no variation in the source carries no source information, and after
    standardisation on the source its target values are divided by an almost-zero spread,
    so it would dominate the transferred score for purely numerical reasons. Such features
    are dropped from the model and from the alignment sum alike, and the count is reported."""
    X = np.asarray(X, dtype=np.float64)
    mu = X.mean(0)
    sd = X.std(0)
    mask = sd > rel * np.maximum(np.abs(mu), 1.0)
    return mu, sd, mask

def transfer_alignment(w, sd_src, sd_tgt, r_tgt):
    """kappa, the alignment between a linear source scorer and the target label.

    For s(x) = sum_j w_j (x_j - mu_j) / sd_src_j over the usable features, the covariance
    with the target label is

        Cov(s, y) = sd_y * sum_j w_j (sd_tgt_j / sd_src_j) r_tgt_j = sd_y * kappa,

    with r_tgt and sd_tgt measured on the data the score is evaluated on. Since sd_y > 0,
    sign(kappa) is the sign of the correlation between transferred scores and target labels,
    so negative kappa means an inverted ranking. Each feature enters weighted by the
    coefficient the source model places on it and by the ratio of spreads, which is why a
    count of sign-reversed features does not predict inversion: a few heavily weighted
    reversals can invert a ranking that many lightly weighted ones would not."""
    w = np.asarray(w, dtype=np.float64).ravel()
    contrib = w * (np.asarray(sd_tgt, dtype=np.float64) / np.asarray(sd_src, dtype=np.float64)) \
              * np.asarray(r_tgt, dtype=np.float64)
    contrib = np.nan_to_num(contrib, nan=0.0, posinf=0.0, neginf=0.0)
    return float(contrib.sum()), contrib

def alignment_share_reversed(contrib):
    """Share of total absolute contribution coming from terms pulling against the label."""
    tot = np.abs(contrib).sum()
    return float(np.abs(contrib[contrib < 0]).sum() / tot) if tot > 0 else np.nan

def safe_corr(a, b):
    """Pearson correlation returning 0 rather than nan when either vector is constant."""
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def col_corr(X, y):
    """Per-column Pearson correlation with a binary label, vectorised."""
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    xm = X.mean(0); ym = y.mean()
    xs = X.std(0); ys = y.std()
    if ys < 1e-12:
        return np.zeros(X.shape[1])
    cov = ((X - xm) * (y - ym)[:, None]).mean(0)
    with np.errstate(invalid='ignore', divide='ignore'):
        r = cov / (xs * ys)
    return np.nan_to_num(r, nan=0.0)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

seed = CFG['seed']
parts, LIN = {}, {}
for tag, d in DATASETS.items():
    tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
    tr = tr.reset_index(drop=True)
    parts[tag] = dict(train=stratified_cap(tr, CFG['train_cap'], seed),
                      eval=stratified_cap(te, CFG['eval_cap'], seed).reset_index(drop=True),
                      train_full=tr)

for tag in CFG['corpora']:
    Xs, med = clean_X(parts[tag]['train'], FEATURES)
    ys = parts[tag]['train']['Label'].values
    mu, sd, mask = usable_features(Xs.values)
    Z = (Xs.values[:, mask] - mu[mask]) / sd[mask]
    t0 = time.time()
    lr = LogisticRegression(max_iter=1000, C=1.0).fit(Z, ys)
    LIN[tag] = dict(w=lr.coef_.ravel(), b=float(lr.intercept_[0]), mu=mu, sd=sd, mask=mask, med=med)
    print(f'{tag}: logistic fit {time.time()-t0:.0f}s, {lr.n_iter_[0]} iters, '
          f'usable features {int(mask.sum())}/{len(FEATURES)}, |w| max {np.abs(lr.coef_).max():.2f}')
    del Xs, Z; gc.collect()

rows, brows = [], []
for s in CFG['corpora']:
    L = LIN[s]; m = L['mask']
    for t in CFG['corpora']:
        if t == s:
            continue
        ev = parts[t]['eval']; yev = ev['Label'].values
        Xev, _ = clean_X(ev, FEATURES, medians=L['med'])
        E = Xev.values[:, m]
        score = ((E - L['mu'][m]) / L['sd'][m]) @ L['w'] + L['b']
        p = 1.0 / (1.0 + np.exp(-np.clip(score, -50, 50)))
        realised = safe_corr(score, yev)

        # oracle form: r and sd measured on exactly the data the score is evaluated on,
        # which makes kappa * sd_y equal to Cov(score, y) and serves as a correctness check
        k_or, contrib = transfer_alignment(L['w'], L['sd'][m], E.std(0), col_corr(E, yev))
        cov = float(np.cov(score, yev, bias=True)[0, 1])
        ident = abs(k_or * yev.std() - cov) / max(abs(cov), 1e-12)

        mm = all_metrics(yev, p)
        bm, bt, bo = best_threshold_two_sided(yev, p)
        rows.append(dict(source=s, target=t, kappa=k_or, identity_rel_err=ident,
                         reversed_share=alignment_share_reversed(contrib),
                         realised_corr=realised, lin_mcc=mm['mcc'], lin_ceiling=bm, lin_orient=bo,
                         n_dropped=int((~m).sum())))
        print(f"  {s}->{t}: kappa {k_or:+.4f}  realised corr {realised:+.4f}  "
              f"identity rel err {ident:.1e}  ceiling {bm:.3f}")

        # deployable form: r and sd from the labelled buffer only
        for b in CFG['budgets']:
            buf = stratified_frac(parts[t]['train_full'], b, seed)
            ybuf = buf['Label'].values
            if len(np.unique(ybuf)) < 2:
                continue
            Xb, _ = clean_X(buf, FEATURES, medians=L['med'])
            B = Xb.values[:, m]
            k_b, cb = transfer_alignment(L['w'], L['sd'][m], B.std(0), col_corr(B, ybuf))
            brows.append(dict(source=s, target=t, budget=b, n_buffer=len(buf),
                              kappa_buffer=k_b, kappa_oracle=k_or, realised_corr=realised,
                              reversed_share=alignment_share_reversed(cb)))
            del Xb
        del Xev, E; gc.collect()

pd.DataFrame(rows).round(6).to_csv(ALIGN_CSV, index=False)
pd.DataFrame(brows).round(6).to_csv(BUF_CSV, index=False)
print('saved', ALIGN_CSV, 'and', BUF_CSV)

In [ ]:
from scipy.stats import spearmanr

A = pd.read_csv(ALIGN_CSV); B = pd.read_csv(BUF_CSV)

print('=== 1. correctness check: kappa reproduces the realised covariance ===')
print(f"  worst relative error of the identity across 12 pairs: {A.identity_rel_err.max():.2e}")
agree = int((np.sign(A.kappa) == np.sign(A.realised_corr)).sum())
print(f"  sign agreement: {agree} of {len(A)} pairs")
r, p = spearmanr(A.kappa, A.realised_corr)
print(f"  Spearman(kappa, realised correlation) = {r:+.3f} (p={p:.2e})")
print(f"  features dropped for no source variation, per corpus: {sorted(A.n_dropped.unique())}")

print('\n=== 2. deployable form: kappa estimated from the labelled buffer ===')
B['sign_ok'] = np.sign(B.kappa_buffer) == np.sign(B.realised_corr)
tab = B.groupby('budget').agg(n=('sign_ok','size'), sign_agreement=('sign_ok','mean'),
                              buffer_rows=('n_buffer','mean')).round(3)
print(tab.to_string())
for b, g in B.groupby('budget'):
    rr, pp = spearmanr(g.kappa_buffer, g.kappa_oracle)
    print(f"  budget {b}: Spearman(kappa from buffer, kappa from full target) = {rr:+.3f} (p={pp:.2e})")

print('\n=== 3. does the linear alignment carry over to the non-linear families? ===')
v3 = pd.read_csv(f'{RESULT}/fc_results_v3.csv')
z = v3[v3.strategy=='zero_shot_2s'].groupby(['source','target']).agg(
        inverted=('orient', lambda s: float((s==-1).mean())), ceiling=('mcc_best_thr','mean')).reset_index()
J = A.merge(z, on=['source','target'])
for col, lab in [('kappa','kappa'), ('reversed_share','reversed share')]:
    r1,p1 = spearmanr(J[col], J.inverted); r2,p2 = spearmanr(J[col], J.ceiling)
    print(f"  {lab:14s} vs tree inversion rate rho={r1:+.3f} (p={p1:.3f}) | vs ceiling rho={r2:+.3f} (p={p2:.3f})")
FT = pd.read_csv(f'{RESULT}/fc_inversion_features.csv')
J2 = J.merge(FT[['source','target','n_flipped','agreement']], on=['source','target'])
r3,p3 = spearmanr(J2.n_flipped, J2.inverted)
print(f"  count of sign-reversed features, for comparison: rho={r3:+.3f} (p={p3:.3f})")
J2.round(4).to_csv(f'{RESULT}/fc_alignment_vs_observed.csv', index=False)
print('\nsaved fc_alignment_vs_observed.csv')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "16: alignment identity for linear transfer (kappa), exact verification, buffer-estimated form"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)